In [36]:
# scripts/run_demo_pipeline.py

# ============================================================
# Demo forecasting pipeline:
# Appliance Energy Prediction
#
# Models:
#   1. Benchmarks
#   2. SARIMAX
#   3. Feature-based model
#   4. Foundation model placeholder
#
# Output:
#   outputs/forecasts/all_forecasts.csv
#   outputs/metrics/model_comparison.csv
#   outputs/figures/forecast_comparison.png
# ============================================================

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.inspection import permutation_importance
from xgboost import XGBRegressor

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
!pip install torch==2.2.2 chronos-forecasting==1.5.3 

In [37]:
__file__ = os.path.join(_dh[0], "run_demo_pipeline.ipynb")

In [38]:
# ------------------------------------------------------------
# 0. Configuration
# ------------------------------------------------------------

RANDOM_STATE = 0
np.random.seed(RANDOM_STATE)

PROJECT_ROOT = Path(__file__).resolve().parents[1]

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

PROCESSED_DIR = DATA_DIR / "processed"
FORECAST_DIR = OUTPUT_DIR / "forecasts"
METRICS_DIR = OUTPUT_DIR / "metrics"
FIGURE_DIR = OUTPUT_DIR / "figures"

for path in [PROCESSED_DIR, FORECAST_DIR, METRICS_DIR, FIGURE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

TARGET = "Appliances"

# Hourly data:
# 24 observations = 1 day
# 168 observations = 1 week
DAILY_PERIOD = 24
WEEKLY_PERIOD = 168

# Use final 14 days as the test set
TEST_STEPS = 14 * 24

# Ranges for the SARIMAX order grid search 
SARIMAX_P_RANGE = range(0, 7)
SARIMAX_D_RANGE = range(0, 3)
SARIMAX_Q_RANGE = range(0, 7)
SARIMAX_SEASONAL_ORDER = (1, 1, 1, DAILY_PERIOD)

ORDER_SEARCH_PATH = METRICS_DIR / "sarimax_order_search.csv"

# Forecast horizon 
HORIZON = 24
N_ORIGINS = TEST_STEPS // HORIZON

In [39]:
# ------------------------------------------------------------
# 1. Load and prepare data
# ------------------------------------------------------------

def load_appliance_data():
    """
    Load Appliances Energy Prediction data from the original UCI CSV.
    
    """

    local_path = DATA_DIR / "raw" / "energydata_complete.csv"

    if local_path.exists():
        print(f"Loading data from local file")
        df = pd.read_csv(local_path)
    else:
        print("Local file not found. Downloading data from UCI...")
        url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00374/energydata_complete.csv"
        df = pd.read_csv(url)

    print("\nColumns in downloaded data:")
    print(df.columns.tolist())

    df["date"] = pd.to_datetime(df["date"])
    df = df.set_index("date").sort_index()

    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=[TARGET])

    print("Original data shape:", df.shape)

    hourly = df.resample("h").mean()
    hourly = hourly.interpolate("time")
    hourly = hourly.dropna()

    print("Hourly data shape:", hourly.shape)

    hourly.to_csv(PROCESSED_DIR / "appliance_hourly.csv")

    return hourly


def load_raw_10min_data():
    """
    Convenience loader for the original, un-resampled 10-minute data.
    """

    local_path = DATA_DIR / "raw" / "energydata_complete.csv"

    if local_path.exists():
        df = pd.read_csv(local_path)
    else:
        url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00374/energydata_complete.csv"
        df = pd.read_csv(url)

    df["date"] = pd.to_datetime(df["date"])
    df = df.set_index("date").sort_index()

    return df

In [40]:
# ------------------------------------------------------------
# 1B. Exploratory data analysis and stationarity checks (Part 1)
# ------------------------------------------------------------

def plot_missing_values(df):
    """
    Check every column for missing values. 
    """

    missing = df.isna().sum()
    missing = missing[missing > 0]
    missing_df = missing.rename("missing_count").to_frame()

    missing_df.to_csv(METRICS_DIR / "missing_values.csv")

    if len(missing) == 0:
        print("\nNo missing values found in any column of the raw data.")
    else:
        print("\nMissing values found:")
        print(missing_df)

    return missing_df


def plot_initial_series(raw_10min, hourly):
    """
     initial plots of the data at both the original 10-minute
    resolution and after resampling to an hourly mean.
    """

    fig, axes = plt.subplots(2, 1, figsize=(14, 8))

    raw_10min[TARGET].plot(ax=axes[0], linewidth=0.3, color="tab:blue")
    axes[0].set_title("Appliances energy use - original 10-minute data")
    axes[0].set_xlabel("Date")
    axes[0].set_ylabel("Wh")

    hourly[TARGET].plot(ax=axes[1], linewidth=0.6, color="tab:orange")
    axes[1].set_title("Appliances energy use - resampled to hourly mean")
    axes[1].set_xlabel("Date")
    axes[1].set_ylabel("Wh")

    fig.tight_layout(pad=2.0)
    fig.savefig(FIGURE_DIR / "eda_raw_and_hourly.png", dpi=200, bbox_inches="tight")


def plot_seasonal_decomposition(y_hourly, period=DAILY_PERIOD, filename="eda_seasonal_decomposition.png"):
    """
    Classical additive decomposition of the hourly Appliances series into
    trend, seasonal and residual components, using a period of 24 to look
    for a daily cycle.
    """

    decomposition = seasonal_decompose(y_hourly, model="additive", period=period)

    fig = decomposition.plot()
    fig.set_size_inches(12, 8)
    fig.suptitle(f"Additive seasonal decomposition of hourly {y_hourly.name} (period = {period})", y=1.02)
    fig.axes[-1].set_xlabel("Date")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / filename, dpi=200, bbox_inches="tight")

    return decomposition


def plot_acf_pacf(series, lags, filename, title_prefix=""):
    """
    Plot the ACF and PACF of a series side by side 
    """

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    plot_acf(series, lags=lags, ax=axes[0])
    axes[0].set_title(f"{title_prefix} ACF".strip())
    axes[0].set_xlabel("Lag (hours)")
    axes[0].set_ylabel("Autocorrelation")

    plot_pacf(series, lags=lags, ax=axes[1], method="ywm")
    axes[1].set_title(f"{title_prefix} PACF".strip())
    axes[1].set_xlabel("Lag (hours)")
    axes[1].set_ylabel("Partial autocorrelation")

    fig.tight_layout(pad=2.0)
    fig.savefig(FIGURE_DIR / filename, dpi=200, bbox_inches="tight")



def print_eda_summary_stats(hourly, target=TARGET):
    """
    EDA: descriptive statistics for the target series, a
    breakdown by hour of day and by day of week, and a correlation
    table against every other numeric column.
    """

    y = hourly[target]

    print(f"\nHourly {target} summary statistics:")
    print(f"  mean   = {y.mean():.1f} Wh")
    print(f"  median = {y.median():.1f} Wh")
    print(f"  std    = {y.std():.1f} Wh")
    print(f"  max    = {y.max():.1f} Wh")

    by_hour = y.groupby(hourly.index.hour).mean()
    quiet_hour = int(by_hour.idxmin())
    busy_hour = int(by_hour.idxmax())
    print(
        f"\nBy hour of day: quietest at {quiet_hour:02d}:00 ({by_hour.min():.1f} Wh), "
        f"busiest at {busy_hour:02d}:00 ({by_hour.max():.1f} Wh)"
    )

    by_dow = y.groupby(hourly.index.day_name()).mean().sort_values(ascending=False)
    print(f"\nBy day of week (highest to lowest):")
    print(by_dow)

    is_weekend = hourly.index.dayofweek >= 5
    weekday_mean = y[~is_weekend].mean()
    weekend_mean = y[is_weekend].mean()
    print(f"\nWeekday average = {weekday_mean:.1f} Wh, weekend average = {weekend_mean:.1f} Wh")

    numeric_cols = hourly.select_dtypes(include=[np.number]).columns.drop(target)
    corr = hourly[numeric_cols].corrwith(y).sort_values(key=lambda s: s.abs(), ascending=False)
    print(f"\nCorrelation with {target} (all variables, strongest first):")
    print(corr)

    summary = pd.Series({
        "mean": y.mean(),
        "median": y.median(),
        "std": y.std(),
        "max": y.max(),
        "quiet_hour": quiet_hour,
        "quiet_hour_value": by_hour.min(),
        "busy_hour": busy_hour,
        "busy_hour_value": by_hour.max(),
        "weekday_mean": weekday_mean,
        "weekend_mean": weekend_mean,
        "top_day": by_dow.index[0],
        "top_day_value": by_dow.iloc[0],
        "bottom_day": by_dow.index[-1],
        "bottom_day_value": by_dow.iloc[-1],
    })
    summary.to_csv(METRICS_DIR / "eda_summary_stats.csv", header=["value"])
    by_dow.to_csv(METRICS_DIR / "eda_day_of_week.csv", header=["mean_wh"])
    corr.to_csv(METRICS_DIR / "eda_correlations.csv", header=["correlation"])

    return summary, by_hour, by_dow, corr


def run_stationarity_tests(series, label=""):
    """
    Run the Augmented Dickey-Fuller test and the KPSS test for
    stationarity on a series.
    """

    series = series.dropna()

    adf_stat, adf_p, _, _, adf_crit, _ = adfuller(series, autolag="AIC")

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        kpss_stat, kpss_p, _, kpss_crit = kpss(series, regression="c", nlags="auto")

    result = {
        "series": label,
        "adf_statistic": adf_stat,
        "adf_p_value": adf_p,
        "adf_1pct_crit": adf_crit["1%"],
        "kpss_statistic": kpss_stat,
        "kpss_p_value": kpss_p,
        "kpss_10pct_crit": kpss_crit["10%"],
    }

    print(f"\nStationarity tests for: {label}")
    print(f"  ADF statistic = {adf_stat:.3f}, p-value = {adf_p:.4f}")
    print(f"  KPSS statistic = {kpss_stat:.3f}, p-value = {kpss_p:.4f}")

    return result

In [41]:
# ------------------------------------------------------------
# 2. Evaluation
# ------------------------------------------------------------

def rmse(y_true, y_pred):
    
    return np.sqrt(mean_squared_error(y_true, y_pred))


def mase(y_true, y_pred, y_train, seasonality=24):
    """
    Mean absolute scaled error.

    Uses the in-sample seasonal naive forecast error as scale.
    """

    y_train = pd.Series(y_train).astype(float)

    seasonal_errors = np.abs(
        y_train.iloc[seasonality:].values
        - y_train.iloc[:-seasonality].values
    )

    scale = seasonal_errors.mean()

    if scale == 0:
        return np.nan

    return np.mean(np.abs(y_true - y_pred)) / scale


def evaluate_forecast(name, y_true, y_pred, y_train):
    y_true = pd.Series(y_true).astype(float)
    y_pred = pd.Series(y_pred, index=y_true.index).astype(float)

    return {
        "model": name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": rmse(y_true, y_pred),
        "MASE": mase(y_true, y_pred, y_train, seasonality=DAILY_PERIOD),
        "Bias": np.mean(y_pred - y_true),
    }

In [42]:
# ------------------------------------------------------------
# 3. Benchmark models
# ------------------------------------------------------------

def mean_forecast(y_train, horizon, index):
    return pd.Series(y_train.mean(), index=index, name="mean")


def naive_forecast(y_train, horizon, index):
    return pd.Series(y_train.iloc[-1], index=index, name="naive")


def seasonal_naive_forecast(y_train, horizon, index, seasonality):
    """
    Recursive seasonal naive forecast.

    For hourly data:
        seasonality=24 gives same hour yesterday.
        seasonality=168 gives same hour last week.
    """

    values = []

    history = list(y_train.values)

    for i in range(horizon):
        values.append(history[-seasonality])
        history.append(values[-1])

    return pd.Series(values, index=index)


def drift_forecast(y_train, horizon, index):
    slope = (y_train.iloc[-1] - y_train.iloc[0]) / (len(y_train) - 1)

    values = [
        y_train.iloc[-1] + slope * step
        for step in range(1, horizon + 1)
    ]

    return pd.Series(values, index=index, name="drift")

In [43]:
# ------------------------------------------------------------
# 4. SARIMAX model
# ------------------------------------------------------------

def fit_sarimax(y_train, X_train=None, order=(1, 0, 1), seasonal_order=(1, 1, 1, 24), maxiter=None):
    """
    Fit a SARIMAX model.
    """

    model = SARIMAX(
        y_train,
        exog=X_train,
        order=order,
        seasonal_order=seasonal_order,
        trend="c",
        enforce_stationarity=False,
        enforce_invertibility=False,
    )

    fit_kwargs = {"disp": False}
    if maxiter is not None:
        fit_kwargs["maxiter"] = maxiter

    fit = model.fit(**fit_kwargs)

    return fit


def forecast_sarimax(fit, horizon, index, X_test=None):
    fc = fit.get_forecast(
        steps=horizon,
        exog=X_test,
    )

    mean = fc.predicted_mean
    mean.index = index
    mean.name = "sarimax"

    return mean


def forecast_sarimax_with_ci(fit, horizon, index, X_test=None, alpha=0.05):
    """
    Same idea as forecast_sarimax, but also returns the confidence
    interval for the forecast 
    """

    fc = fit.get_forecast(steps=horizon, exog=X_test)

    mean = fc.predicted_mean
    mean.index = index
    mean.name = "sarimax"

    ci = fc.conf_int(alpha=alpha)
    ci.index = index
    ci.columns = ["sarimax_lower", "sarimax_upper"]

    return mean, ci


def grid_search_sarimax_order(
    y_train,
    p_range=SARIMAX_P_RANGE,
    d_range=SARIMAX_D_RANGE,
    q_range=SARIMAX_Q_RANGE,
    checkpoint_path=ORDER_SEARCH_PATH,
    max_new_fits=None,
):
    """
    this searches the non-seasonal order (p, d, q) and
    keeps the seasonal order fixed at SARIMAX_SEASONAL_ORDER = (1, 1, 1, 24).
    
    """

    combos = [
        (p, d, q)
        for p in p_range
        for d in d_range
        for q in q_range
    ]

    if checkpoint_path.exists():
        done = pd.read_csv(checkpoint_path)
        done_set = set(zip(done["p"], done["d"], done["q"]))
        rows = done.to_dict("records")
    else:
        done_set = set()
        rows = []

    n_new = 0

    for (p, d, q) in combos:
        if (p, d, q) in done_set:
            continue

        if max_new_fits is not None and n_new >= max_new_fits:
            break

        try:
            model = SARIMAX(
                y_train,
                order=(p, d, q),
                seasonal_order=(0, 0, 0, 0),
                trend="c" if d == 0 else None,
                enforce_stationarity=False,
                enforce_invertibility=False,
            )
            fit = model.fit(disp=False, maxiter=100)
            aic = fit.aic
            converged = bool(fit.mle_retvals.get("converged", True))
        except Exception:
            aic = np.nan
            converged = False

        rows.append({"p": p, "d": d, "q": q, "aic": aic, "converged": converged})
        n_new += 1

        pd.DataFrame(rows).to_csv(checkpoint_path, index=False)

    results = pd.DataFrame(rows)
    remaining = len(combos) - len(results)

    if remaining > 0:
        print(
            f"\nSARIMAX order search: {len(results)}/{len(combos)} "
            f"combinations complete, {remaining} remaining."
        )
    else:
        print(f"\nSARIMAX order search complete: all {len(results)} combinations fitted.")

    valid = results.dropna(subset=["aic"])

    if len(valid) == 0:
        print("No combination converged; falling back to default order (1, 0, 1).")
        return (1, 0, 1), results

    best_row = valid.loc[valid["aic"].idxmin()]
    best_order = (int(best_row["p"]), int(best_row["d"]), int(best_row["q"]))

    print(f"Best non-seasonal order by AIC: {best_order} (AIC = {best_row['aic']:.1f})")

    return best_order, results


def plot_sarimax_residual_diagnostics(fit, filename="residual_acf.png"):
    """
    assess model fit by inspecting the residuals. Produces the
    residual ACF (to check autocorrelation has been captured), a
    histogram of the residuals, and a Q-Q plot (to check they are
    roughly normally distributed, as SARIMAX assumes). Also runs a
    Ljung-Box test at lag 24 as a formal check for remaining
    autocorrelation at the daily seasonal lag.
    """

    resid = fit.resid.dropna()

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    plot_acf(resid, lags=48, ax=axes[0])
    axes[0].set_title("Residual ACF")
    axes[0].set_xlabel("Lag (hours)")
    axes[0].set_ylabel("Autocorrelation")

    axes[1].hist(resid, bins=40, color="tab:blue", edgecolor="white")
    axes[1].set_title("Residual distribution")
    axes[1].set_xlabel("Residual (Wh)")
    axes[1].set_ylabel("Count")

    stats.probplot(resid, dist="norm", plot=axes[2])
    axes[2].set_title("Residual Q-Q plot")

    fig.tight_layout(pad=2.0)
    fig.savefig(FIGURE_DIR / filename, dpi=200, bbox_inches="tight")


    ljung = acorr_ljungbox(resid, lags=[24], return_df=True)
    ljung.to_csv(METRICS_DIR / "sarimax_residual_ljungbox.csv")

    print("\nLjung-Box test on SARIMAX residuals (lag 24):")
    print(ljung)

    return resid, ljung


def plot_sarimax_forecast_with_ci(test, sarimax_mean, sarimax_ci, filename="sarimax_forecast_ci.png"):
    """plot the SARIMAX forecast against the actual test data with
    its 95% confidence interval shaded around it.."""

    fig, ax = plt.subplots(figsize=(14, 6))

    test.plot(ax=ax, label="Actual", color="black", linewidth=1.5)
    sarimax_mean.plot(ax=ax, label="SARIMAX forecast", color="tab:red")

    ax.fill_between(
        test.index,
        sarimax_ci["sarimax_lower"],
        sarimax_ci["sarimax_upper"],
        color="tab:red",
        alpha=0.2,
        label="95% confidence interval",
    )

    ax.set_title("SARIMAX forecast with 95% confidence interval (first 24-hour origin)")
    ax.set_ylabel("Appliance energy use (Wh)")
    ax.set_xlabel("Date")
    ax.legend()

    fig.tight_layout()
    fig.savefig(FIGURE_DIR / filename, dpi=200, bbox_inches="tight")
    # plt.close(fig) removed on purpose, see plot_initial_series() above.

In [44]:
# ------------------------------------------------------------
# 5. Feature engineering and the feature-based model (XGBoost)
# ------------------------------------------------------------

def add_time_features(df):
    out = df.copy()

    out["hour"] = out.index.hour
    out["dayofweek"] = out.index.dayofweek
    out["is_weekend"] = (out["dayofweek"] >= 5).astype(int)

    out["hour_sin"] = np.sin(2 * np.pi * out["hour"] / 24)
    out["hour_cos"] = np.cos(2 * np.pi * out["hour"] / 24)

    out["dow_sin"] = np.sin(2 * np.pi * out["dayofweek"] / 7)
    out["dow_cos"] = np.cos(2 * np.pi * out["dayofweek"] / 7)

    return out


def make_ml_table(df, target=TARGET, horizon=HORIZON, test_steps=TEST_STEPS):
    """
    Build the supervised-learning table for the feature-based model.

    """

    out = add_time_features(df)
    n = len(out)

    for lag in [24, 48, 168]:
        out[f"lag_{lag}"] = out[target].shift(lag)

    block_id = (np.arange(n) - (n - test_steps)) // horizon
    out["_block_id"] = block_id

    y = out[target]
    block_start_positions = pd.Series(np.arange(n), index=block_id).groupby(level=0).min()

    for window in [24, 168]:
        block_mean = {}
        block_std = {}
        for b, start_pos in block_start_positions.items():
            window_slice = y.iloc[max(0, start_pos - window):start_pos]
            if len(window_slice) == window:
                block_mean[b] = window_slice.mean()
                block_std[b] = window_slice.std()
            else:
                block_mean[b] = np.nan
                block_std[b] = np.nan
        out[f"roll_mean_{window}"] = out["_block_id"].map(block_mean)
        out[f"roll_std_{window}"] = out["_block_id"].map(block_std)

    out = out.drop(columns=["_block_id"])

    return out.dropna()


def fit_feature_model(X_train, y_train, params=None):
    """
    Feature-based model, using XGBoost's XGBRegressor.

    """

    if params is None:
        params = {"n_estimators": 500, "learning_rate": 0.05, "max_depth": 5}

    model = XGBRegressor(
        random_state=RANDOM_STATE,
        n_jobs=-1,
        **params,
    )

    model.fit(X_train, y_train)

    return model


def tune_feature_model(X_train, y_train, n_val_steps=TEST_STEPS):
    """
    hyperparameter search for XGBoost.
    """

    X_fit = X_train.iloc[:-n_val_steps]
    y_fit = y_train.iloc[:-n_val_steps]
    X_val = X_train.iloc[-n_val_steps:]
    y_val = y_train.iloc[-n_val_steps:]

    param_grid = [
        {"n_estimators": 200, "learning_rate": 0.05, "max_depth": 3},
        {"n_estimators": 500, "learning_rate": 0.05, "max_depth": 3},
        {"n_estimators": 500, "learning_rate": 0.1, "max_depth": 3},
        {"n_estimators": 500, "learning_rate": 0.05, "max_depth": 5},
        {"n_estimators": 500, "learning_rate": 0.05, "max_depth": 7},
        {"n_estimators": 1000, "learning_rate": 0.01, "max_depth": 5},
    ]

    rows = []

    for params in param_grid:
        model = XGBRegressor(random_state=RANDOM_STATE, n_jobs=-1, **params)
        model.fit(X_fit, y_fit)
        val_pred = model.predict(X_val)
        val_score = rmse(y_val, val_pred)

        if val_score is None or (isinstance(val_score, float) and np.isnan(val_score)):
            raise RuntimeError(
                f"tune_feature_model: rmse() returned {val_score!r} for "
                f"params {params}. This should not happen - if you see "
                "this, try Kernel > Restart Kernel and Run All, then "
                "re-run."
            )

        rows.append({**params, "val_rmse": val_score})

    results = pd.DataFrame(rows).sort_values("val_rmse").reset_index(drop=True)
    results.to_csv(METRICS_DIR / "feature_model_tuning.csv", index=False)

    valid_results = results.dropna(subset=["val_rmse"])
    if len(valid_results) == 0:
        raise RuntimeError(
            "tune_feature_model: every candidate in the hyperparameter "
            "grid failed to produce a validation RMSE. See "
            "outputs/metrics/feature_model_tuning.csv for what was "
            "recorded for each combination."
        )

    best = valid_results.iloc[0]
    best_params = {
        "n_estimators": int(best["n_estimators"]),
        "learning_rate": float(best["learning_rate"]),
        "max_depth": int(best["max_depth"]),
    }

    print("\nFeature model hyperparameter search:")
    print(results)
    print(f"Best params: {best_params} (validation RMSE = {float(best['val_rmse']):.2f})")

    return best_params, results


def forecast_feature_model(model, X_test, index):
    pred = model.predict(X_test)

    return pd.Series(
        pred,
        index=index,
        name="feature_model",
    )


def plot_feature_importance(model, X_test, y_test, filename="feature_importance.png", csv_filename="feature_importance.csv", n_top=15, n_repeats=5):
    """
    XGBoost has its own built-in (gain-based) feature
    importances, but permutation importance is used here instead, so
    that the feature model can be inspected the same way regardless of
    which library fits it
    """

    result = permutation_importance(
        model,
        X_test,
        y_test,
        n_repeats=n_repeats,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

    importance_df = pd.DataFrame({
        "feature": X_test.columns,
        "importance_mean": result.importances_mean,
        "importance_std": result.importances_std,
    }).sort_values("importance_mean", ascending=False)

    importance_df.to_csv(METRICS_DIR / csv_filename, index=False)

    top = importance_df.head(n_top).iloc[::-1]

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.barh(top["feature"], top["importance_mean"], xerr=top["importance_std"], color="tab:green")
    ax.set_xlabel("Permutation importance (drop in R$^2$)")
    ax.set_ylabel("Feature")
    ax.set_title("Feature-based model: top feature importances")

    fig.tight_layout(pad=2.0)
    fig.savefig(FIGURE_DIR / filename, dpi=200, bbox_inches="tight")

    return importance_df

In [45]:
# ------------------------------------------------------------
# 6. Foundation model
# ------------------------------------------------------------

def load_chronos_pipeline():
    """
    Load the Chronos pipeline once.
    """

    from chronos import ChronosPipeline
    import torch

    return ChronosPipeline.from_pretrained(
        "amazon/chronos-t5-tiny",
        device_map="cpu",
        torch_dtype=torch.float32,
    )


def forecast_foundation_model_chronos(pipeline, y_train, horizon, index):
    """
    a zero-shot forecast using Amazon's Chronos
    time-series foundation model, given an already-loaded pipeline (see
    load_chronos_pipeline()).

    horizon is meant to be 24, one daily origin at a time, not the full
    336-hour test period in one call. 
    """

    import torch

    context = torch.tensor(y_train.values, dtype=torch.float32)

    forecast = pipeline.predict(
        context=context,
        prediction_length=horizon,
        num_samples=100,
    )

    median = np.median(forecast[0].numpy(), axis=0)

    return pd.Series(median, index=index, name="foundation_model")


def forecast_foundation_model_timegpt(y_train, horizon, index):
    """
     zero-shot forecast using Nixtla's TimeGPT, a
    time-series foundation model reached over Nixtla's hosted API rather
    than downloaded and run locally. 
    """

    import os
    from nixtla import NixtlaClient

    api_key = os.environ.get("NIXTLA_API_KEY")
    if not api_key:
        raise RuntimeError("NIXTLA_API_KEY environment variable is not set.")

    client = NixtlaClient(api_key=api_key)

    df = pd.DataFrame({"ds": y_train.index, "y": y_train.values})
    forecast_df = client.forecast(df=df, h=horizon, freq="h")

    values = forecast_df["TimeGPT"].values[:horizon]

    return pd.Series(values, index=index, name="foundation_model")


def get_foundation_model_forecast_rolling(
    train, test, horizon=HORIZON, n_origins=N_ORIGINS, note_filename="foundation_model_notes.txt"
):
    """
    Foundation-model forecast for the rolling backtest: 14 separate
    24-hour-ahead forecasts, one per daily origin, using the real,
    expanding history at each origin - the same design as the
    benchmarks and the rolling SARIMAX forecast (see rolling_backtest()'s
    docstring). 
    """

    cache_path = METRICS_DIR / "foundation_forecast_cache.csv"
    expected_index = test.index[: n_origins * horizon]
    if cache_path.exists():
        cached = pd.read_csv(cache_path, index_col=0, parse_dates=True)["foundation_model"]
        if len(cached) == len(expected_index) and cached.index.equals(expected_index):
            note_path = METRICS_DIR / note_filename
            if note_path.exists():
                print("\n" + note_path.read_text() + " (loaded from cache)")
            return cached

    failures = []

    try:
        pipeline = load_chronos_pipeline()
        blocks = []
        history = train.copy()
        for origin in range(n_origins):
            print(f"  Chronos: forecasting origin {origin + 1}/{n_origins}...", flush=True)
            block_index = test.index[origin * horizon: (origin + 1) * horizon]
            blocks.append(forecast_foundation_model_chronos(pipeline, history, horizon, block_index))
            history = pd.concat([history, test.loc[block_index]])
        forecast = pd.concat(blocks).rename("foundation_model")
        note = (
            f"Foundation model used: Chronos (amazon/chronos-t5-tiny), zero-shot, "
            f"{n_origins} daily 24-hour-ahead forecasts using the real expanding history at each origin."
        )
        print("\n" + note)
        (METRICS_DIR / note_filename).write_text(note)
        forecast.to_frame().to_csv(cache_path)
        return forecast
    except Exception as exc:
        reason = f"Chronos (amazon/chronos-t5-tiny): {type(exc).__name__}: {exc}"
        failures.append(reason)
        print(f"\nCould not run Chronos here ({type(exc).__name__}: {exc}). Trying the next option.")

    try:
        blocks = []
        history = train.copy()
        for origin in range(n_origins):
            print(f"  TimeGPT: forecasting origin {origin + 1}/{n_origins}...", flush=True)
            block_index = test.index[origin * horizon: (origin + 1) * horizon]
            blocks.append(forecast_foundation_model_timegpt(history, horizon, block_index))
            history = pd.concat([history, test.loc[block_index]])
        forecast = pd.concat(blocks).rename("foundation_model")
        note = (
            f"Foundation model used: TimeGPT (Nixtla API), zero-shot, "
            f"{n_origins} daily 24-hour-ahead forecasts using the real expanding history at each origin."
        )
        print("\n" + note)
        (METRICS_DIR / note_filename).write_text(note)
        forecast.to_frame().to_csv(cache_path)
        return forecast
    except Exception as exc:
        reason = f"TimeGPT (Nixtla API): {type(exc).__name__}: {exc}"
        failures.append(reason)
        print(f"\nCould not run TimeGPT here ({type(exc).__name__}: {exc}). Trying the next option.")

    blocks = []
    history = train.copy()
    for origin in range(n_origins):
        block_index = test.index[origin * horizon: (origin + 1) * horizon]
        blocks.append(seasonal_naive_forecast(history, horizon, block_index, seasonality=DAILY_PERIOD))
        history = pd.concat([history, test.loc[block_index]])
    forecast = pd.concat(blocks).rename("foundation_model")
    note = (
        "Foundation model requested: Chronos, then TimeGPT. Neither could "
        "run in this environment:\n"
        + "\n".join(f"  - {f}" for f in failures)
        + "\nFalling back to the daily seasonal naive forecast as a "
        "documented substitute. Both attempts are recorded here rather "
        "than skipped, so the failure is honest instead of silent."
    )
    print("\n" + note)
    (METRICS_DIR / note_filename).write_text(note)
    forecast.to_frame().to_csv(cache_path)

    return forecast

In [46]:
# ------------------------------------------------------------
# 7. Plotting and summaries
# ------------------------------------------------------------

def plot_forecasts(train, test, forecast_df, title="Appliance energy forecasting"):
    fig, ax = plt.subplots(figsize=(14, 7))

    # Plot final part of training data for context
    train.tail(14 * 24).plot(
        ax=ax,
        label="Training data",
        linewidth=1.5,
    )

    test.plot(
        ax=ax,
        label="Test data",
        linewidth=2.0,
        color="black",
    )

    for col in forecast_df.columns:
        if col in ("actual", "sarimax_lower", "sarimax_upper"):
            continue
        forecast_df[col].plot(
            ax=ax,
            label=col,
            alpha=0.9,
        )

    ax.set_title(title)
    ax.set_ylabel("Appliance energy use (Wh)")
    ax.set_xlabel("Date")
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0), borderaxespad=0.0, fontsize=9)

    fig.tight_layout(rect=[0, 0, 0.86, 1])

    return fig


def plot_error_diagnostics(forecast_df, filename="error_diagnostics.png", title_suffix=""):
    """
   two extra diagnostic views on top of the main
    forecast_comparison plot. Left panel: forecast error (prediction
    minus actual) over the test period for every model, which shows
    whether errors grow across the horizon or stay roughly constant.
    Right panel: distribution of those errors, which shows whether a
    model is systematically biased (box shifted away from zero) as well
    as how spread out its errors are.
    """

    model_cols = [
        c for c in forecast_df.columns
        if c not in ("actual", "sarimax_lower", "sarimax_upper")
    ]

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    for col in model_cols:
        error = forecast_df[col] - forecast_df["actual"]
        error.plot(ax=axes[0], label=col, alpha=0.8)

    axes[0].axhline(0, color="black", linewidth=1)
    axes[0].set_title(f"Forecast error over the test period{title_suffix}")
    axes[0].set_xlabel("Date")
    axes[0].set_ylabel("Predicted - actual (Wh)")
    axes[0].legend(fontsize=8)

    errors = [(forecast_df[col] - forecast_df["actual"]).dropna() for col in model_cols]
    axes[1].boxplot(errors, labels=model_cols, showfliers=False)
    axes[1].axhline(0, color="black", linewidth=1)
    axes[1].set_title(f"Distribution of forecast errors{title_suffix}")
    axes[1].set_xlabel("Model")
    axes[1].set_ylabel("Predicted - actual (Wh)")
    axes[1].tick_params(axis="x", rotation=45)

    fig.tight_layout(pad=2.0)
    fig.savefig(FIGURE_DIR / filename, dpi=200, bbox_inches="tight")



def summarise_model_comparison(results_df, output_filename="model_comparison_summary.txt"):
    """
    identify the strongest benchmark model, then work
    out how much each of the more complex models improves (or fails to
    improve) on it, using MASE since it is scale-free. 
    """

    benchmark_names = ["mean", "naive", "seasonal_naive_daily", "seasonal_naive_weekly", "drift"]

    benchmarks = results_df[results_df["model"].isin(benchmark_names)]
    best_benchmark = benchmarks.sort_values("MASE").iloc[0]

    lines = [
        f"Strongest benchmark: {best_benchmark['model']} "
        f"(MASE={best_benchmark['MASE']:.3f}, RMSE={best_benchmark['RMSE']:.2f}, "
        f"MAE={best_benchmark['MAE']:.2f}, Bias={best_benchmark['Bias']:.2f})",
    ]

    for name in ["sarimax", "feature_model", "foundation_model"]:
        if name in results_df["model"].values:
            row = results_df[results_df["model"] == name].iloc[0]
            change = (row["MASE"] - best_benchmark["MASE"]) / best_benchmark["MASE"] * 100
            direction = "worse than" if change > 0 else "better than"
            lines.append(
                f"{name}: MASE={row['MASE']:.3f}, RMSE={row['RMSE']:.2f}, "
                f"MAE={row['MAE']:.2f}, Bias={row['Bias']:.2f} "
                f"-> {abs(change):.1f}% {direction} the strongest benchmark"
            )

    summary_text = "\n".join(lines)

    print("\nModel comparison summary (vs strongest benchmark):")
    print(summary_text)

    (METRICS_DIR / output_filename).write_text(summary_text)

    return summary_text


# ------------------------------------------------------------
# 7B. 24-hour-ahead rolling backtest 
# ------------------------------------------------------------

def rolling_backtest(
    data,
    sarimax_fit,
    exog_cols,
    ml_params,
    horizon=HORIZON,
    n_origins=N_ORIGINS,
    test_steps=TEST_STEPS,
    foundation_forecast=None,
):
    """
    Walk-forward evaluation with a genuine 24-hour horizon, repeated at
    14 daily origins across the 336-hour test period.

    """

    y = data[TARGET]
    train = y.iloc[:-test_steps]
    test = y.iloc[-test_steps:]

    X_all = data[exog_cols] if exog_cols else None

    # ---- SARIMAX: roll forward the already-fitted model with refit=False ----
    sarimax_blocks = []

    for origin in range(n_origins):
        block_start = origin * horizon
        block_index = test.index[block_start: block_start + horizon]

        X_block = X_all.loc[block_index] if X_all is not None else None

        fc = sarimax_fit.get_forecast(steps=horizon, exog=X_block)
        block_pred = fc.predicted_mean
        block_pred.index = block_index
        sarimax_blocks.append(block_pred)


        y_actual_block = test.loc[block_index]
        sarimax_fit = sarimax_fit.append(y_actual_block, exog=X_block, refit=False)

    rolling_forecasts = {"sarimax": pd.concat(sarimax_blocks).rename("sarimax")}

    # ---- Benchmarks ----
    benchmark_blocks = {
        "mean": [], "naive": [], "seasonal_naive_daily": [],
        "seasonal_naive_weekly": [], "drift": [],
    }
    history = train.copy()

    for origin in range(n_origins):
        block_start = origin * horizon
        block_index = test.index[block_start: block_start + horizon]

        benchmark_blocks["mean"].append(mean_forecast(history, horizon, block_index))
        benchmark_blocks["naive"].append(naive_forecast(history, horizon, block_index))
        benchmark_blocks["seasonal_naive_daily"].append(
            seasonal_naive_forecast(history, horizon, block_index, seasonality=DAILY_PERIOD)
        )
        benchmark_blocks["seasonal_naive_weekly"].append(
            seasonal_naive_forecast(history, horizon, block_index, seasonality=WEEKLY_PERIOD)
        )
        benchmark_blocks["drift"].append(drift_forecast(history, horizon, block_index))

        history = pd.concat([history, test.loc[block_index]])

    for name, blocks in benchmark_blocks.items():
        rolling_forecasts[name] = pd.concat(blocks).rename(name)

    # ---- Feature-based model----
    ml_data = make_ml_table(data, target=TARGET)
    ml_feature_cols = [c for c in ml_data.columns if c != TARGET]

    ml_train = ml_data.iloc[:-test_steps]
    ml_test = ml_data.iloc[-test_steps:]

    feature_model = fit_feature_model(
        X_train=ml_train[ml_feature_cols],
        y_train=ml_train[TARGET],
        params=ml_params,
    )

    rolling_forecasts["feature_model"] = forecast_feature_model(
        model=feature_model,
        X_test=ml_test[ml_feature_cols],
        index=ml_test.index,
    ).reindex(test.index)

    importance_df = plot_feature_importance(
        model=feature_model,
        X_test=ml_test[ml_feature_cols],
        y_test=ml_test[TARGET],
    )

    # ---- Foundation model ----
    if foundation_forecast is None:
        foundation_forecast = get_foundation_model_forecast_rolling(
            train=train, test=test, horizon=horizon, n_origins=n_origins,
        )
    rolling_forecasts["foundation_model"] = foundation_forecast

    # ---- Evaluate ----
    results = []

    for name, pred in rolling_forecasts.items():
        pred = pred.reindex(test.index)
        valid = pred.notna() & test.notna()

        results.append(
            evaluate_forecast(
                name=name,
                y_true=test.loc[valid],
                y_pred=pred.loc[valid],
                y_train=train,
            )
        )

    results_df = (
        pd.DataFrame(results)
        .sort_values("MASE")
        .reset_index(drop=True)
    )

    print("\n24-hour rolling backtest - model comparison:")
    print(results_df.round(3))

    forecast_df = pd.DataFrame({"actual": test})
    for name, pred in rolling_forecasts.items():
        forecast_df[name] = pred.reindex(test.index)

    forecast_df.to_csv(FORECAST_DIR / "all_forecasts.csv")
    results_df.to_csv(METRICS_DIR / "model_comparison.csv", index=False)

    return results_df, forecast_df, feature_model, importance_df

In [ ]:
# ------------------------------------------------------------
# 8. Main pipeline
# ------------------------------------------------------------

def run_pipeline():
    # --------------------------------------------------------
    # Load data
    # --------------------------------------------------------

    data = load_appliance_data()

    # --------------------------------------------------------
    # Part 1: EDA and stationarity checks
    # --------------------------------------------------------

    raw_10min = load_raw_10min_data()

    plot_missing_values(raw_10min)
    plot_initial_series(raw_10min, data)
    plot_seasonal_decomposition(data[TARGET], period=DAILY_PERIOD)
    plot_acf_pacf(
        data[TARGET],
        lags=72,
        filename="eda_acf_pacf_level.png",
        title_prefix="Appliances (level)",
    )
    print_eda_summary_stats(data)

    stationarity_rows = [
        run_stationarity_tests(data[TARGET], label="Appliances (level)"),
        run_stationarity_tests(data[TARGET].diff(), label="Appliances (first difference)"),
    ]
    pd.DataFrame(stationarity_rows).to_csv(METRICS_DIR / "stationarity_tests.csv", index=False)

    # --------------------------------------------------------
    # Part 2: forecasting problem definition / train-test split
    #
    #   Target variable    : Appliances (hourly mean, Wh)
    #   Forecast horizon   : 24 hours, evaluated at 14 daily origins
    #                        across the 336-hour test period (Parts
    #                        3/4/6/7, README "Forecasting task") - see
    #                        rolling_backtest() below.
    #   Train/test split   : final 14 days held out as test, everything
    #                        before that used as training data
    #   Evaluation metrics : MAE, RMSE, MASE (scaled by the in-sample
    #                        daily seasonal naive error), Bias
    # --------------------------------------------------------

    y = data[TARGET]

    train = y.iloc[:-TEST_STEPS]
    test = y.iloc[-TEST_STEPS:]

    print("\nTrain period:")
    print(train.index.min(), "to", train.index.max())

    print("\nTest period:")
    print(test.index.min(), "to", test.index.max())

    # --------------------------------------------------------
    # SARIMAX with selected exogenous variables
    # --------------------------------------------------------

    candidate_exog_cols = [
        "T_out",
        "RH_out",
        "Windspeed",
        "Visibility",
        "Tdewpoint",
    ]

    exog_cols = [
        col for col in candidate_exog_cols
        if col in data.columns
    ]

    if len(exog_cols) > 0:
        X = data[exog_cols]
        X_train = X.iloc[:-TEST_STEPS]

        print("\nSARIMAX exogenous columns:")
        print(exog_cols)
        print(
            "Note: at each daily origin, the forecast for the next 24 hours "
            "uses the realised (actual) weather for those hours, not a "
            "weather forecast. "
        )
    else:
        X = None
        X_train = None

        print("\nNo SARIMAX exogenous columns found. Fitting target-only SARIMAX.")

    best_order, order_search_results = grid_search_sarimax_order(y_train=train)

    final_order = (1, 0, 3)

    sarimax_fit = fit_sarimax(
        y_train=train,
        X_train=X_train,
        order=final_order,
        seasonal_order=SARIMAX_SEASONAL_ORDER,
        maxiter=30,
    )

    sarimax_coefficients = sarimax_fit.params.rename("coefficient").to_frame()
    sarimax_coefficients["std_err"] = sarimax_fit.bse
    sarimax_coefficients["p_value"] = sarimax_fit.pvalues
    sarimax_coefficients.to_csv(METRICS_DIR / "sarimax_coefficients.csv")
    print("\nSARIMAX fitted coefficients:")
    print(sarimax_coefficients.round(4))

    plot_sarimax_residual_diagnostics(sarimax_fit, filename="residual_acf.png")

    first_block_index = test.index[:HORIZON]
    X_first_block = X.loc[first_block_index] if X is not None else None

    first_mean, first_ci = forecast_sarimax_with_ci(
        fit=sarimax_fit,
        horizon=HORIZON,
        index=first_block_index,
        X_test=X_first_block,
    )
    plot_sarimax_forecast_with_ci(test.loc[first_block_index], first_mean, first_ci)

    # --------------------------------------------------------
    # Feature-based model
    # --------------------------------------------------------

    ml_data = make_ml_table(data, target=TARGET)
    ml_train = ml_data.iloc[:-TEST_STEPS]
    feature_cols = [col for col in ml_data.columns if col != TARGET]

    best_ml_params, ml_tuning_results = tune_feature_model(
        X_train=ml_train[feature_cols],
        y_train=ml_train[TARGET],
    )

    # --------------------------------------------------------
    # benchmarks, SARIMAX, the feature model and the
    # foundation model, all evaluated as 14 daily 24-hour-ahead forecasts
    # --------------------------------------------------------

    results_df, forecast_df, feature_model, importance_df = rolling_backtest(
        data=data,
        sarimax_fit=sarimax_fit,
        exog_cols=exog_cols,
        ml_params=best_ml_params,
    )

    summarise_model_comparison(results_df)

    # --------------------------------------------------------
    # Save plots
    # --------------------------------------------------------

    fig = plot_forecasts(
        train=train,
        test=test,
        forecast_df=forecast_df,
        title="Appliance energy forecasting - 24-hour rolling backtest",
    )

    fig.savefig(
        FIGURE_DIR / "forecast_comparison.png",
        dpi=300,
        bbox_inches="tight",
    )
    # plt.close(fig) removed on purpose, see plot_initial_series() above.

    plot_error_diagnostics(forecast_df)

    print("\nSaved outputs:")
    print("all_forecasts.csv")
    print("model_comparison.csv")
    print("forecast_comparison.png")
    print("error_diagnostics.png")
    print("feature_importance.png")

    return results_df, forecast_df, importance_df


if __name__ == "__main__":
    run_pipeline()

Loading data from local file

Columns in downloaded data:
['date', 'Appliances', 'lights', 'T1', 'RH_1', 'T2', 'RH_2', 'T3', 'RH_3', 'T4', 'RH_4', 'T5', 'RH_5', 'T6', 'RH_6', 'T7', 'RH_7', 'T8', 'RH_8', 'T9', 'RH_9', 'T_out', 'Press_mm_hg', 'RH_out', 'Windspeed', 'Visibility', 'Tdewpoint', 'rv1', 'rv2']
Original data shape: (19735, 28)
Hourly data shape: (3290, 28)

No missing values found in any column of the raw data.

Hourly Appliances summary statistics:
  mean   = 97.8 Wh
  median = 63.3 Wh
  std    = 81.2 Wh
  max    = 608.3 Wh

By hour of day: quietest at 03:00 (48.2 Wh), busiest at 18:00 (191.8 Wh)

By day of week (highest to lowest):
date
Monday       111.450684
Saturday     106.246345
Friday       105.192982
Sunday        94.915936
Thursday      90.434028
Wednesday     89.930556
Tuesday       87.125000
Name: Appliances, dtype: float64

Weekday average = 96.7 Wh, weekend average = 100.6 Wh

Correlation with Appliances (all variables, strongest first):
lights         0.261794
R